# 01 — Procedencia de pilotos y sensibilidad con curvas propias de GAITANA

El objetivo es evaluar una curva construida con pilotos atribuibles a GAITANA, sin ampliar el modelo a ARABELLA. Las estadísticas procesadas no contienen finca explícita. Los números de bloque se repiten entre fincas: nunca se asigna finca sólo por bloque.

Se cruza **bloque + cama + identidad comercial exacta + fecha de siembra**, contra los planos conservados de 2021–2026. Se exige una única finca en toda la evidencia. La primera fecha de plano coincidente se conserva y sólo permite usar el piloto desde esa fecha, para no incorporar una identificación conocida después del origen. La atribución sigue siendo inferida, no certificada por la empresa. Sin coincidencia o con más de una finca, queda pendiente.

Se conserva la curva compartida original y se calcula una alternativa con la misma ventana de 78 semanas, edad, regla IQR y horizonte. No se inventan curvas ni se rellenan faltantes. El uso de planos posteriores en la comprobación de ambigüedad sólo puede excluir una atribución, nunca incorporar producción futura. Esta sensibilidad requiere confirmar físicamente las correspondencias antes de uso operativo.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Datos_analiticos_proyecto').exists())
PROY=ROOT/'Notebooks/Proyecciones Teoricas Propias/resultados_todos_los_anios'
OUT=ROOT/'Datos_analiticos_proyecto/revision_clima_finca';OUT.mkdir(exist_ok=True)
ANIOS=list(range(2021,2027))
def normalizar(s):return s.astype('string').str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('ascii').str.strip().str.upper().str.replace(r'\s+',' ',regex=True)
def numero(s):
    texto=normalizar(s).str.replace(r'^BLOQUE\s*','',regex=True)
    valor=pd.to_numeric(texto,errors='coerce')
    entero=valor.notna()&valor.mod(1).eq(0)
    texto.loc[entero]=valor.loc[entero].astype('Int64').astype('string')
    return texto

parts=[]
for anio in ANIOS:
    p=pd.read_parquet(PROY/str(anio)/'planos.parquet')
    p['bloque_union']=numero(p.invernadero);p['cama_union']=numero(p.cama)
    parts.append(p[['bloque_union','cama_union','clave_curva','fecha_siembra','finca','fecha_plano']].drop_duplicates())
planos=pd.concat(parts,ignore_index=True)
keys=['bloque_union','cama_union','clave_curva','fecha_siembra']
validos=planos.dropna(subset=keys+['finca'])
mapa=validos.groupby(keys,as_index=False).agg(n_fincas=('finca','nunique'),fincas=('finca',lambda s:';'.join(sorted(s.unique()))),primera_evidencia=('fecha_plano','min'))
mapa['finca_inferida']=mapa.fincas.where(mapa.n_fincas.eq(1))
mapa.to_parquet(OUT/'mapa_finca_pilotos.parquet',index=False)
print('Claves observadas:',len(mapa),'ambiguas entre fincas:',int(mapa.n_fincas.gt(1).sum()))
display(validos.groupby('bloque_union').finca.nunique().value_counts().rename('bloques'))


Claves observadas: 68563 ambiguas entre fincas: 0


finca
1    23
2    16
3     2
Name: bloques, dtype: int64

In [2]:
resumen=[];referencias=[];alt=[];comparacion=[]
for anio in ANIOS:
    pilotos=pd.read_parquet(PROY/str(anio)/'camas_piloto.parquet')
    pilotos['bloque_union']=numero(pilotos.bloque);pilotos['cama_union']=numero(pilotos.cama)
    atrib=pilotos.merge(mapa,on=keys,how='left',validate='many_to_one')
    atrib['estado_finca']=np.select([atrib.n_fincas.isna(),atrib.n_fincas.gt(1)],['sin_pareja','ambigua'],default='unica_inferida')
    referencias.append(atrib[['archivo','id_piloto','fila_excel','referencia','bloque','cama','clave_curva','fecha_siembra','finca_inferida','primera_evidencia','estado_finca']].assign(anio=anio))
    u=pd.read_parquet(PROY/str(anio)/'estadisticas.parquet')
    u=u.merge(atrib[['id_piloto','finca_inferida','primera_evidencia','estado_finca']],on='id_piloto',how='left',validate='many_to_one')
    resumen.append(u.groupby(['finca_inferida','estado_finca'],dropna=False).agg(observaciones=('id_estadistica','size'),pilotos=('id_piloto','nunique')).reset_index().assign(anio=anio))
    q=pd.read_parquet(PROY/str(anio)/'proyecciones_teoricas.parquet')
    q=q.loc[q.version_seleccionada&q.finca.eq('GAITANA')].copy()
    for fecha,g in q.groupby('fecha_plano'):
        h=u.loc[u.finca_inferida.eq('GAITANA')&u.primera_evidencia.le(fecha)&u.fecha_corte.lt(fecha)&u.fecha_corte.ge(fecha-pd.Timedelta(weeks=78))].copy()
        assert h.empty or h.fecha_corte.max()<fecha
        h['r']=h.corte/h.plantas
        stats=h.groupby(['clave_curva','edad']).r
        quant=stats.quantile([.25,.75]).unstack().rename(columns={.25:'q1',.75:'q3'})
        quant['n']=stats.size()
        h=h.join(quant,on=['clave_curva','edad'],validate='many_to_one')
        li=h.q1-1.5*(h.q3-h.q1);ls=h.q3+1.5*(h.q3-h.q1)
        h['rc']=h.r.where(h.n.lt(4),h.r.clip(lower=li,upper=ls))
        cv=h.groupby(['clave_curva','edad'],as_index=False).agg(r_local=('rc','mean'),n_local=('rc','size'))
        g=g.merge(cv,on=['clave_curva','edad'],how='left',validate='many_to_one')
        impedido=g.estado.isin(['plantas_invalidas','sin_fecha_siembra','siembra_posterior','sin_variedad_maestra'])
        g['teoria_local']=np.where(g.estado.eq('siembra_posterior'),0,np.where(~impedido&g.r_local.notna(),g.plantas*g.r_local,np.nan))
        g['sin_cobertura_local']=g.teoria_local.isna().astype(int)
        assert len(g)==len(q.loc[q.fecha_plano.eq(fecha)])
        for col in ['finca','producto','color','variedad']:g['union_'+col]=normalizar(g[col])
        dims=['plano','semana_proyeccion']+['union_'+col for col in ['finca','producto','color','variedad']]
        z=g.groupby(dims,dropna=False).agg(teoria_local=('teoria_local',lambda x:x.sum(min_count=1)),sin_cobertura_local=('sin_cobertura_local','sum'),cohortes_local=('estado','size')).reset_index()
        alt.append(z)
        comparacion.append({'anio':anio,'fecha_plano':fecha,'observaciones_gaitana_disponibles':len(h),'cohortes_horizonte':len(g),'plantas':g.plantas.sum(),'plantas_curva_compartida':g.loc[g.estado.eq('proyectada'),'plantas'].sum(),'plantas_curva_local':g.loc[g.teoria_local.notna()&~g.estado.eq('siembra_posterior'),'plantas'].sum()})
    print(anio,'atribuido y proyectado',flush=True)
resumen=pd.concat(resumen,ignore_index=True);referencias=pd.concat(referencias,ignore_index=True)
teoria=pd.concat(alt,ignore_index=True);control=pd.DataFrame(comparacion)
assert not teoria.duplicated(dims).any()
teoria.to_parquet(OUT/'proyecciones_gaitana_inferida.parquet',index=False)
referencias.to_parquet(OUT/'procedencia_pilotos.parquet',index=False)
with pd.ExcelWriter(OUT/'auditoria_finca_curvas.xlsx') as w:
    resumen.to_excel(w,sheet_name='Procedencia_observaciones',index=False)
    control.to_excel(w,sheet_name='Cobertura_por_plano',index=False)
    mapa.loc[mapa.n_fincas.gt(1)].to_excel(w,sheet_name='Claves_ambiguas',index=False)
display(resumen)
annual=control.groupby('anio')[['plantas','plantas_curva_compartida','plantas_curva_local']].sum()
annual['cobertura_compartida']=annual.plantas_curva_compartida/annual.plantas
annual['cobertura_local']=annual.plantas_curva_local/annual.plantas
display(annual)
print('No se reemplazó la proyección compartida. La comparación de modelos evalúa esta alternativa en los mismos casos; no se afirma procedencia certificada.')

2021 atribuido y proyectado


2022 atribuido y proyectado


2023 atribuido y proyectado


2024 atribuido y proyectado


2025 atribuido y proyectado


2026 atribuido y proyectado


,finca_inferida,estado_finca,observaciones,pilotos,anio
0,GAITANA,unica_inferida,46389,1498,2021
1,<NA>,sin_pareja,158932,4110,2021
2,GAITANA,unica_inferida,92820,2546,2022
3,<NA>,sin_pareja,164725,4300,2022
4,GAITANA,unica_inferida,136098,3870,2023
5,<NA>,sin_pareja,180087,4820,2023
6,GAITANA,unica_inferida,186353,5087,2024
7,<NA>,sin_pareja,193207,5186,2024
8,GAITANA,unica_inferida,211395,5907,2025
9,<NA>,sin_pareja,199188,5302,2025


,plantas,plantas_curva_compartida,plantas_curva_local,cobertura_compartida,cobertura_local
anio,,,,,
2021,4.439285e+09,3.601688e+09,3.339758e+09,0.811322,0.752319
2022,4.373579e+09,3.635559e+09,3.528621e+09,0.831255,0.806804
2023,4.488285e+09,3.759173e+09,3.474244e+09,0.837552,0.774069
2024,4.548607e+09,4.101260e+09,3.523602e+09,0.901652,0.774655
2025,4.619860e+09,3.874096e+09,3.400239e+09,0.838574,0.736005
2026,2.955185e+09,2.338127e+09,2.210022e+09,0.791195,0.747846


No se reemplazó la proyección compartida. La comparación de modelos evalúa esta alternativa en los mismos casos; no se afirma procedencia certificada.
